In [4]:
!pip install optuna mlflow dagshub seaborn

  Using cached mlflow-3.10.0-py3-none-any.whl.metadata (31 kB)
  Using cached dagshub-0.6.7-py3-none-any.whl.metadata (12 kB)
  Using cached mlflow_skinny-3.10.0-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.10.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached databricks_sdk-0.91.0-py3-none-any.whl.metadata (40 kB)
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Using cached gql-4.0.0-py3-none-any.whl.metadata (10 kB)
  Using cached dagshub_annotation_converter-0.1.16-py3-none-any.whl.metadata (3.2 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
Using cached mlflow-3.10.0-py3-none-any.whl (10.2 MB)
Using cached mlflow_skinny-3.10.0-py3-none-any.whl (3.0 MB)
Using cach

In [5]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import dagshub
import mlflow
import mlflow.sklearn

In [6]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=85b2a326-facb-49cd-83a1-84484bd1a0ef&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=76801858623563507a01f0298af7fd783f82a3302a7a708bda76a64cf4484829




Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [7]:
df = pd.read_csv("/kaggle/input/datasets/aryanumri/twitter-sentiment/twitter_cleaned (2).csv")

df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [8]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [9]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (41292, 8000)
Test shape: (10323, 8000)


In [10]:
def build_nb(trial):

    alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)

    return MultinomialNB(alpha=alpha)

In [11]:
def objective(trial):

    model = build_nb(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params({f"model_{k}": v for k, v in trial.params.items()})
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [12]:
study = optuna.create_study(
    direction="maximize",
    study_name="multinomial_nb_study",
    storage="sqlite:///multinomial_nb_optuna.db",
    load_if_exists=True
)

[I 2026-02-23 09:46:23,444] A new study created in RDB with name: multinomial_nb_study


In [13]:
with mlflow.start_run(run_name="MultinomialNB"):

    # Log vectorizer parameters safely
    mlflow.log_param("model_name", "MultinomialNB")
    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_ngram_range", "(1,2)")
    mlflow.log_param("vectorizer_max_features", 8000)
    mlflow.log_param("vectorizer_min_df", 2)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("num_features", X_train.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params({f"model_{k}": v for k, v in best_params.items()})
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = MultinomialNB(**best_params)
    final_model.fit(X_train, y_train)

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = MultinomialNB(**best_params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_nb.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_nb.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - MultinomialNB")
    plt.savefig("confusion_matrix_nb.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_nb.png")

    study.trials_dataframe().to_csv("optuna_trials_nb.csv", index=False)
    mlflow.log_artifact("optuna_trials_nb.csv")

    mlflow.sklearn.log_model(final_model, artifact_path="model")

print("MultinomialNB experiment completed successfully.")

🏃 View run trial_0 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/a79bdd36ad38455ab9bce2dacd05a71a
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:46:28,029] Trial 0 finished with value: 0.7211102053314269 and parameters: {'alpha': 0.007207831303456892}. Best is trial 0 with value: 0.7211102053314269.


🏃 View run trial_1 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/9aa4466621f643bdb5f049e8bc545f1c
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:46:35,011] Trial 1 finished with value: 0.7168376384979546 and parameters: {'alpha': 0.21079758768829124}. Best is trial 0 with value: 0.7211102053314269.


🏃 View run trial_2 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/a6adae2a55a14aa9ad23b7c2c245c278
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


...

🏃 View run trial_49 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/8e937fc93e9c44beb0cb34c92d404da8
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:52:11,011] Trial 49 finished with value: 0.7218045035803856 and parameters: {'alpha': 0.002352615526638924}. Best is trial 28 with value: 0.7219011219378736.


Best Macro F1: 0.7219011219378736
Best Params: {'alpha': 0.0020809446205401413}
Fold 1 - Macro F1: 0.7243
Fold 2 - Macro F1: 0.7278
Fold 3 - Macro F1: 0.7340
Fold 4 - Macro F1: 0.7381
Fold 5 - Macro F1: 0.7310
Final Test Macro F1: 0.7219011219378736


2026/02/23 09:52:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 09:52:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run MultinomialNB at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/f4fbdfe1b3054442b3ae4423d6008dc3
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
MultinomialNB experiment completed successfully.
